# Phase 2 dose sweep, two phase-1 adapters

At twenty-five steps, evidence recall was 0.29 to 0.94 on the `augustopt`
adapter and 0.00 to 0.35 on the `origanchors` one. Same dose, same
learning rate, same evidence. The only difference is the phase-1 weights
underneath.

That matters for what the `ft` result can claim. "Absorbed the evidence
and still cannot use it" needs recall to be high. If recall on
`origanchors` never rises with dose, the `ft` zero on that adapter is an
absorption failure and the usability claim rests only on the five
high-dose payloads from the earlier run.

This sweeps dose on one seed for both adapters and asks two things:

* where does recall cross into memorisation on each
* at the dose where recall is high, is `ft` still zero

August's one-world sweep without any phase-1 adapter is the third
reference: recall 0 at 15 steps, 0.19 at 25 with zero of four replies
JSON, and 1.00 at 50 and 100 with still zero of four JSON. The collapse
into emitting other prompts is what phase 1 was introduced to prevent.

In [ ]:
import os
# Trainer wraps the model in DataParallel when it sees two devices, which
# halves the step count and doubles the effective batch. Pin to one.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
%pip install -q -U transformers peft bitsandbytes accelerate

## Preflight

In [ ]:
import sys, glob, json, time, re, collections
import torch

def find_dir(marker, root="/kaggle/input"):
    hits = sorted(glob.glob(os.path.join(root, "**", marker), recursive=True),
                  key=lambda p: (p.count(os.sep), len(p)))
    if not hits:
        raise SystemExit(f"no {marker} under {root}")
    return os.path.dirname(hits[0])

RUN_TAG    = "dose"
ADAPTERS   = ["origanchors", "augustopt"]   # phase-1 weights to compare
DOSES      = [5, 15, 25, 50, 100, 200]
SEEDS      = [7]              # add more only if the session has room

REPO_PATH = find_dir("resource_mdp.py")
EVAL_PATH = find_dir("anchors_v22.py")
PAY_CHANGED = sorted(glob.glob("/kaggle/input/**/payloads_silent_break_det",
                               recursive=True))[0]
OUT_DIR = "/kaggle/working"

paths = {}
for tag in ADAPTERS:
    hits = [h for h in glob.glob("/kaggle/input/**/adapter_config.json",
                                 recursive=True) if tag in h]
    if len(hits) != 1:
        raise SystemExit(f"need exactly one adapter matching {tag!r}, got {hits}")
    paths[tag] = os.path.dirname(hits[0])
    print(f"{tag:14} {paths[tag]}")

assert torch.cuda.is_available(), "no GPU: set Accelerator in the sidebar"
assert torch.cuda.device_count() == 1, (
    "CUDA_VISIBLE_DEVICES did not take; restart the kernel and run from the "
    "first cell")
CAP = torch.cuda.get_device_capability(0)
USE_BF16 = CAP[0] >= 8          # T4 is 7.5 and only emulates bf16
DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print(f"\ngpu: {torch.cuda.get_device_name(0)} | dtype: {DTYPE}")

MODEL_NAME    = "Qwen/Qwen2.5-1.5B-Instruct"
PHASE2_LR     = 2e-4
MAX_LEN       = 2048
RECALL_CUE    = "Recall the observation log for this network."
RECALL_TOKENS = 1600

sys.path.insert(0, EVAL_PATH)
import ecpm_eval as E
E.attach(REPO_PATH)
payloads = {p["seed"]: p for p in
            E.load_payloads(PAY_CHANGED, seeds=set(SEEDS))}
print(f"seeds: {sorted(payloads)}")
print(f"{len(ADAPTERS)} adapters x {len(DOSES)} doses x {len(SEEDS)} seeds "
      f"= {len(ADAPTERS)*len(DOSES)*len(SEEDS)} trainings, "
      f"{sum(DOSES)*len(ADAPTERS)*len(SEEDS)} steps total")

## Evidence and recall

In [ ]:
TRIPLE = re.compile(r"\([A-Z], a\d+, [A-Z]\)")

def evidence_of(pay):
    full = pay["single"][pay["probes"][0]]
    marker = "Observations, period B:"
    tail = full.split(marker, 1)[1]
    body = tail.split("\n\n", 1)[0] if "\n\n" in tail else tail
    return full[:full.index(marker)] + marker + body

def recall_of(pay, text):
    want = set(TRIPLE.findall(evidence_of(pay)))
    got = set(TRIPLE.findall(text or ""))
    return len(want & got) / len(want) if want else None

ev = evidence_of(payloads[SEEDS[0]])
print(f"seed {SEEDS[0]} evidence: {len(ev)} chars, "
      f"{len(set(TRIPLE.findall(ev)))} distinct triples")

## Model

In [ ]:
from transformers import (AutoModelForCausalLM, AutoTokenizer,
                          BitsAndBytesConfig, Trainer, TrainingArguments)
from peft import PeftModel, prepare_model_for_kbit_training
from torch.utils.data import Dataset

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=DTYPE,
                         bnb_4bit_use_double_quant=True)
base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb, device_map={"": 0})
base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True)

def as_ids(x):
    if hasattr(x, "input_ids"):
        x = x.input_ids
    elif hasattr(x, "keys") and "input_ids" in x.keys():
        x = x["input_ids"]
    x = list(x)
    if x and isinstance(x[0], (list, tuple)):
        x = list(x[0])
    if not all(isinstance(t, int) for t in x):
        raise TypeError(f"expected token ids, got {type(x[0]).__name__}")
    return x

def chat_example(user, answer):
    pre = as_ids(tok.apply_chat_template(
        [{"role": "system", "content": E.SYSTEM},
         {"role": "user", "content": user}],
        add_generation_prompt=True, tokenize=True))
    ans = as_ids(tok(answer + tok.eos_token,
                     add_special_tokens=False)["input_ids"])
    return {"input_ids": pre + ans, "labels": [-100] * len(pre) + ans}

class OneShot(Dataset):
    def __init__(self, row, n):
        self.rows = [row] * max(n, 1)
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        return dict(self.rows[i])

def collate(batch):
    n = max(len(b["input_ids"]) for b in batch)
    pad = tok.pad_token_id
    out = {"input_ids": [], "labels": [], "attention_mask": []}
    for b in batch:
        gap = n - len(b["input_ids"])
        out["input_ids"].append(b["input_ids"] + [pad] * gap)
        out["labels"].append(b["labels"] + [-100] * gap)
        out["attention_mask"].append([1] * len(b["input_ids"]) + [0] * gap)
    return {k: torch.tensor(v) for k, v in out.items()}

def generate(model, messages, max_new_tokens):
    e = tok.apply_chat_template(messages, add_generation_prompt=True,
                                return_tensors="pt", return_dict=True)
    e = {k: v.to(model.device) for k, v in e.items()}
    with torch.no_grad():
        o = model.generate(**e, max_new_tokens=max_new_tokens,
                           do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(o[0, e["input_ids"].shape[1]:], skip_special_tokens=True)

def fresh_adapter(path):
    m = PeftModel.from_pretrained(base, path, is_trainable=True)
    assert sum(p.numel() for p in m.parameters() if p.requires_grad) > 0, (
        "no trainable parameters: PeftModel needs is_trainable=True")
    return m

print("ready | GPU GB:", round(torch.cuda.memory_allocated() / 1e9, 2))

## Sweep

Dose zero is the phase-1 adapter untouched, which is `arm_c` with an
empty prompt and `arm_c` proper. It anchors the curve.

In [ ]:
rows_path = os.path.join(OUT_DIR, f"raw_{RUN_TAG}.jsonl")
rows, done = [], set()
if os.path.exists(rows_path):
    for line in open(rows_path):
        r = json.loads(line)
        rows.append(r)
        done.add((r["adapter"], r["dose"], r["seed"], r["arm"], r["probe"]))
    print(f"resuming, {len(rows)} rows already saved")

sink = open(rows_path, "a")
for tag in ADAPTERS:
    for dose in [0] + DOSES:
        for seed in SEEDS:
            if (tag, dose, seed, "ft", "detection") in done:
                print(f"{tag} dose {dose} seed {seed}: done")
                continue
            pay = payloads[seed]
            t0 = time.time()
            model = fresh_adapter(paths[tag])
            if dose > 0:
                model.config.use_cache = False
                row = chat_example(RECALL_CUE, evidence_of(pay))
                assert len(row["input_ids"]) <= MAX_LEN
                Trainer(
                    model=model,
                    args=TrainingArguments(
                        output_dir=f"{OUT_DIR}/dose_tmp",
                        per_device_train_batch_size=1,
                        gradient_accumulation_steps=1,
                        max_steps=dose, learning_rate=PHASE2_LR,
                        warmup_steps=1, lr_scheduler_type="cosine",
                        logging_strategy="no", save_strategy="no",
                        report_to=[], bf16=USE_BF16, fp16=not USE_BF16,
                        gradient_checkpointing=True,
                        gradient_checkpointing_kwargs={"use_reentrant": False}),
                    train_dataset=OneShot(row, dose),
                    data_collator=collate).train()
            model.eval()
            model.config.use_cache = True
            rec = recall_of(pay, generate(
                model, [{"role": "system", "content": E.SYSTEM},
                        {"role": "user", "content": RECALL_CUE}],
                RECALL_TOKENS))
            new = []
            new += E.run_arm([pay], lambda m, n: generate(model, m, n),
                             arm="ft", mode="single", with_evidence=False,
                             verbose=False)
            new += E.run_arm([pay], lambda m, n: generate(model, m, n),
                             arm="combined", mode="single",
                             with_evidence=True, verbose=False)
            for r in new:
                r["adapter"], r["dose"], r["evidence_recall"] = tag, dose, rec
                sink.write(json.dumps(r) + "\n")
            sink.flush()
            rows += new
            base = model.unload()
            del model
            torch.cuda.empty_cache()
            print(f"{tag:12} dose {dose:<4} seed {seed}: recall {rec:.2f}, "
                  f"{time.time()-t0:.0f}s")
sink.close()
print(f"\n{len(rows)} rows")

## Recall against dose

The question is whether `origanchors` ever absorbs the log, and if so at
what dose. If it crosses 0.9 and `ft` is still zero there, the usability
claim holds on this adapter too and no longer rests on the earlier
high-dose run alone.

In [ ]:
import pandas as pd

def cell(tag, dose, probe, arm):
    m = [r for r in rows if r["adapter"] == tag and r["dose"] == dose
         and r["arm"] == arm and r["probe"] == probe]
    return m[0] if m else None

recall_tbl, loc_tbl = {}, {}
for tag in ADAPTERS:
    recall_tbl[tag] = {}
    loc_tbl[tag] = {}
    for dose in [0] + DOSES:
        r = cell(tag, dose, "detection", "ft")
        recall_tbl[tag][dose] = round(r["evidence_recall"], 2) if r else None
        out = []
        for arm in ("ft", "combined"):
            c = cell(tag, dose, "localization", arm)
            out.append("-" if c is None
                       else ("OK" if c["scored"].get("correct") else "x"))
        loc_tbl[tag][dose] = "/".join(out)

print("evidence recall by dose")
display(pd.DataFrame(recall_tbl).T)
print("\nlocalization, ft / combined  (OK = correct)")
display(pd.DataFrame(loc_tbl).T)

print("\nAugust, no phase-1 adapter: recall 0 at 15, 0.19 at 25, "
      "1.00 at 50 and 100, and zero of four JSON at every dose")

In [ ]:
print("format holds at every dose?")
for tag in ADAPTERS:
    for dose in [0] + DOSES:
        a = [r for r in rows if r["adapter"] == tag and r["dose"] == dose]
        if not a:
            continue
        bare = sum(bool(r["bare_json"]) for r in a)
        ok = sum(1 for r in a if r["parsed"].get("status") == "ok")
        print(f"  {tag:12} dose {dose:<4} bare JSON {bare}/{len(a)}   "
              f"parsed ok {ok}/{len(a)}")

print("\nft action labels by dose (legal labels look like aK):")
LEGAL = re.compile(r"^a\d+$")
for tag in ADAPTERS:
    said = []
    for dose in [0] + DOSES:
        c = cell(tag, dose, "localization", "ft")
        if c and c["parsed"]["status"] == "ok":
            lab = str(c["parsed"].get("action"))
            said.append(f"{dose}:{lab}{'*' if LEGAL.match(lab) else ''}")
    print(f"  {tag:12} {'  '.join(said)}")

## Package

In [ ]:
import zipfile, shutil
shutil.rmtree(f"{OUT_DIR}/dose_tmp", ignore_errors=True)
ZIP_PATH = f"{OUT_DIR}/{RUN_TAG}_all.zip"
n = 0
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for path in sorted(glob.glob(f"{OUT_DIR}/**/*", recursive=True)):
        if (not os.path.isfile(path) or os.path.basename(ZIP_PATH) in path
                or ".ipynb_checkpoints" in path or "adapter_model" in path):
            continue
        z.write(path, os.path.relpath(path, OUT_DIR))
        n += 1
print(f"{n} files -> {ZIP_PATH} ({os.path.getsize(ZIP_PATH)/1e6:.1f} MB)")